# 03 — Train CNN

Train ResNet/VGG on spectrogram features.

### Training Setup

Run the next code cell to train with tuned hyperparameters, class-bias mitigation, and checkpoint saving (`best.pt`, `last.pt`, `epoch_XXX.pt`).


In [ ]:
from config import TrainConfig
from train import run_training

cfg = TrainConfig()
cfg.dataset_root = "./data/ravdess"
cfg.split_json = "./splits/split_seed42.json"

cfg.out_dir = "./outputs/exp1_tuned"
cfg.model_name = "resnet18"  # or "vgg11_bn"

cfg.epochs = 30
cfg.batch_size = 32

cfg.lr = 8e-4
cfg.weight_decay = 2e-4

# STFT ablation: choose one setting from 02_STFT_Spectrogram.ipynb
cfg.n_fft = 512
cfg.win_length = 400
cfg.hop_length = 160

# Improve robustness for small validation set and class bias
cfg.class_weight_mode = "sqrt_inv_freq"
cfg.label_smoothing = 0.05
cfg.lr_scheduler = "cosine"
cfg.min_lr = 1e-5
cfg.selection_metric = "val_acc_ema"
cfg.val_ema_alpha = 0.6
cfg.save_every_epoch = True
cfg.save_last = True

results = run_training(cfg)
results


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
hist = results["history"]
train_loss = np.array(hist["train_loss"], dtype=np.float32)
val_loss = np.array(hist["val_loss"], dtype=np.float32)
train_acc = np.array(hist["train_acc"], dtype=np.float32)
val_acc = np.array(hist["val_acc"], dtype=np.float32)
val_acc_ema = np.array(hist.get("val_acc_ema", hist["val_acc"]), dtype=np.float32)
epochs = np.arange(1, len(train_loss) + 1)
best_idx = int(np.argmax(val_acc))
best_val = float(val_acc[best_idx])
final_train = float(train_acc[-1])
final_val = float(val_acc[-1])
generalization_gap = final_train - final_val
val_std = float(np.std(val_acc))
val_ema_gain = float(val_acc_ema[-1] - val_acc_ema[0])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, train_loss, label="train loss")
axes[0].plot(epochs, val_loss, label="val loss")
axes[0].set_title("Loss Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[1].plot(epochs, train_acc, label="train acc")
axes[1].plot(epochs, val_acc, label="val acc")
axes[1].plot(epochs, val_acc_ema, "--", label="val acc (EMA)")
axes[1].scatter([best_idx + 1], [best_val], color="red", label=f"best val={best_val:.3f}")
axes[1].set_title("Accuracy Curves")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()
print(f"Best checkpoint path: {results['best_path']}")
print(f"Best val acc: {best_val:.4f} (epoch {best_idx + 1})")
print(f"Final train acc: {final_train:.4f}")
print(f"Final val acc: {final_val:.4f}")
print(f"Generalization gap (train-val): {generalization_gap:.4f}")
print(f"Val acc std (stability): {val_std:.4f}")
print(f"Val EMA gain (trend): {val_ema_gain:.4f}")
quality_tags = []
if best_val >= 0.75:
    quality_tags.append("Validation accuracy is strong.")
elif best_val >= 0.65:
    quality_tags.append("Validation accuracy is moderate and usable.")
else:
    quality_tags.append("Validation accuracy is still limited.")
if generalization_gap <= 0.12:
    quality_tags.append("Generalization gap is controlled (low overfitting risk).")
elif generalization_gap <= 0.2:
    quality_tags.append("Generalization gap is noticeable; monitor overfitting.")
else:
    quality_tags.append("Generalization gap is high; overfitting is likely.")
if val_std <= 0.08:
    quality_tags.append("Validation curve is relatively stable.")
else:
    quality_tags.append("Validation curve is noisy (small val set effect).")
if val_ema_gain > 0:
    quality_tags.append("Smoothed validation trend is improving.")
else:
    quality_tags.append("Smoothed validation trend is flat or declining.")
print("\nTraining quality diagnosis:")
for i, t in enumerate(quality_tags, 1):
    print(f"{i}. {t}")